# Pandas Practice — Home Credit Default Risk (zero → analyst-ready)

A full practice workbook that starts from the absolute basics and builds, with deliberate repetition, up to the pandas an analyst uses to engineer model-ready features for credit risk. Built on the Home Credit tables — the same data as your matplotlib and SQL worksheets, so patterns reinforce across all three.

**How to use this notebook**

1. Work top to bottom. Nothing later assumes a skill you haven't met yet. Each section: brief context and key syntax → **one worked example you run** → exercises you write yourself in the empty cells → interpretation questions you answer in markdown.
2. Repetition is the point. Early sections drill the same 4–5 moves (select, filter, group, aggregate) on different columns until they're automatic. Don't skip them because they look easy — speed and correctness come from reps.
3. Every section has **interpretation questions**. In credit risk, saying what a number *means for the portfolio* matters as much as producing it.
4. The later sections (merging, window ops, the capstone) rebuild the exact feature table from the SQL worksheet — do both and compare; that's how the two skills lock in.

**The tables** (grain = what one row means)

| file | grain | key(s) |
|---|---|---|
| `application_train` | one **current application** (has `TARGET`) | `SK_ID_CURR` |
| `bureau` | one **external credit** at the bureau | `SK_ID_BUREAU`, links via `SK_ID_CURR` |
| `previous_application` | one **prior Home Credit application** | `SK_ID_PREV`, via `SK_ID_CURR` |
| `installments_payments` | one **installment payment** | `SK_ID_PREV` / `SK_ID_CURR` |
| `pos_cash_balance` | one **month** of a prior POS/cash loan | `SK_ID_PREV` / `SK_ID_CURR` |

`TARGET = 1` = payment difficulties (default). Because it's 0/1, **the mean of `TARGET` is the default rate** — you'll lean on that constantly. `DAYS_*` columns are **negative** day-counts from the application date.

## 0 — Setup and data loading

Run these. We load `application_train` once into `df` and reuse it throughout. The child tables (`bureau`, etc.) are large — later sections load them with `usecols` to stay fast.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("pandas:", pd.__version__)

df = pd.read_csv(f"../../Datasets/Home Credit Default Risk/application_train.csv")
print("application_train:", df.shape)
df.head(3)


pandas: 3.0.3
application_train: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,...,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,...,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,...,0.0,0.0,0.0,-815.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


### Derived columns (build once, reuse everywhere)

Just like the matplotlib notebook, create clean derived columns up front. The `DAYS_*` fields are negative; `DAYS_EMPLOYED` carries the `365243` sentinel (~1000 years) for "not employed" — we'll deal with it properly in Section 7, so leave it *dirty* here.

In [34]:
df["AGE_YEARS"]           = -df["DAYS_BIRTH"] / 365.25
df["YEARS_EMPLOYED"]      = -df["DAYS_EMPLOYED"] / 365.25   # still dirty: sentinel -> ~ -999.98
df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"]  / df["AMT_INCOME_TOTAL"]
df["ANNUITY_INCOME_RATIO"]= df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]

print("baseline default rate:", round(df["TARGET"].mean(), 4))
df[["AGE_YEARS", "YEARS_EMPLOYED", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO"]].describe()


baseline default rate: 0.0807


C:\Users\User\AppData\Local\Temp\ipykernel_15132\1817577557.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["AGE_YEARS"]           = -df["DAYS_BIRTH"] / 365.25
C:\Users\User\AppData\Local\Temp\ipykernel_15132\1817577557.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["YEARS_EMPLOYED"]      = -df["DAYS_EMPLOYED"] / 365.25   # still dirty: sentinel -> ~ -999.98
C:\Users\User\AppData\Local\Temp\ipykernel_15132\1817577557.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `

,AGE_YEARS,YEARS_EMPLOYED,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO
count,307511.000000,307511.000000,307511.000000,307499.000000
mean,43.906900,-174.716074,3.957570,0.180930
std,11.947950,386.791969,2.689728,0.094574
min,20.503765,-999.980835,0.004808,0.000224
25%,33.984942,0.791239,2.018667,0.114782
50%,43.121150,3.321013,3.265067,0.162833
75%,53.886379,7.556468,5.159880,0.229067
max,69.073238,49.040383,84.736842,1.875965


## 1 — The Series: pandas' 1-D building block

A **Series** is a 1-D labelled array — a single column. It has *values* and an *index* (the labels). Every DataFrame column is a Series. Operations are **vectorised**: you write `s * 2`, not a loop.

**Key syntax**

```python
s = df["AMT_CREDIT"]        # a column IS a Series
s.mean(); s.max(); s.min()  # scalar reductions
s.head(); s.shape; s.dtype  # inspection
s * 1.1                     # vectorised: applies to every element
s.index                     # the row labels
```

**Worked example**:

In [ ]:
s = df["AMT_CREDIT"]
print(type(s))
print("dtype:", s.dtype, " shape:", s.shape)
print("mean:", s.mean(), " max:", s.max())
(s / 1000).head()            # vectorised: credit in thousands


### Exercises

**1.1** — Pull `AMT_INCOME_TOTAL` into a Series `inc`. Print its `mean`, `median`, `min`, `max`.

**1.2** — Create a new Series = `AMT_ANNUITY` divided by 12 (monthly annuity). Show the first 5 values.

**1.3** — From `df["TARGET"]`, compute the mean. Confirm it equals the baseline default rate you printed in Section 0. Explain in a comment why the mean of a 0/1 column is a rate.

**1.4** — Print the `dtype` of `df["CODE_GENDER"]` and of `df["AMT_CREDIT"]`. Why are they different?

**Interpretation 1.1** — "Vectorised" — in one sentence, why is `s * 2` preferred over a Python `for` loop over the values?

In [2]:
# Ex 1.1

inc = df['AMT_INCOME_TOTAL']

inc.head(5)

print(inc.mean())
print(inc.median())
print(inc.min())
print(inc.max())

168797.9192969845
147150.0
25650.0
117000000.0


In [7]:
# Ex 1.2

df['AMT_MONTHLY_ANNUITY'] = df['AMT_ANNUITY'] / 12

print(df['AMT_ANNUITY'].head(5))

df['AMT_MONTHLY_ANNUITY'].head(5)


0    24700.5
1    35698.5
2     6750.0
3    29686.5
4    21865.5
Name: AMT_ANNUITY, dtype: float64


0    2058.375
1    2974.875
2     562.500
3    2473.875
4    1822.125
Name: AMT_MONTHLY_ANNUITY, dtype: float64

In [12]:
# Ex 1.3

print(f"{df['TARGET'].mean():.2%}")

round(df['TARGET'].mean() * 100, 2)


# mean of this column is just the average of a bunch of 0's and 1's added together divded by the total count. 1 for defualt, 0 for no default. The mean decribes the percentage of borrowers that have defaaulter i.e. have a '1'.

8.07%


np.float64(8.07)

In [16]:
# Ex 1.4

print(df['CODE_GENDER'].dtype)

print(df['AMT_CREDIT'].dtype)

#  One holds text the other a float (non integer number) 

str
float64


**Your answer (1.1):**

Less processing power. Easier to read, easier to code.

...

## 2 — DataFrame anatomy

A **DataFrame** is a 2-D table: a dict of Series sharing one index. Learn its inspection methods cold — you run them at the start of every analysis.

**Key syntax**

```python
df.shape            # (rows, cols)
df.columns          # column labels
df.dtypes           # type of each column
df.info()           # dtypes + non-null counts + memory
df.describe()       # numeric summary stats
df.head(); df.tail(); df.sample(5)
df["col"].value_counts()          # category frequencies
```

**Worked example**:

In [ ]:
print(df.shape)
print(df.dtypes.value_counts())    # how many cols of each type
df.describe()[["AMT_INCOME_TOTAL", "AMT_CREDIT", "AGE_YEARS"]]


### Exercises

**2.1** — Print `df.shape`. How many rows and columns?

**2.2** — Use `df.info()`. Which of `AMT_ANNUITY`, `AMT_GOODS_PRICE`, `OCCUPATION_TYPE` have missing values (non-null count < total rows)?

**2.3** — `value_counts()` on `NAME_INCOME_TYPE`. Then again with `normalize=True` to get proportions.

**2.4** — `describe()` the columns `["AGE_YEARS", "CREDIT_INCOME_RATIO", "CNT_CHILDREN"]`. Note anything that looks off (e.g. an implausible max).

**Interpretation 2.1** — From 2.4: `CREDIT_INCOME_RATIO` has a very large max. Is that necessarily an error, or could it be real? What one follow-up check would you run?

In [ ]:
# Ex 2.1

df.shape

# 307511 rows and 123 columns

(307511, 123)

In [21]:
# Ex 2.2

# df.info()

df['AMT_ANNUITY'].info()

df['AMT_GOODS_PRICE'].info()

df['OCCUPATION_TYPE'].info()

# All three have missing values

<class 'pandas.Series'>
RangeIndex: 307511 entries, 0 to 307510
Series name: AMT_ANNUITY
Non-Null Count   Dtype  
--------------   -----  
307499 non-null  float64
dtypes: float64(1)
memory usage: 2.3 MB
<class 'pandas.Series'>
RangeIndex: 307511 entries, 0 to 307510
Series name: AMT_GOODS_PRICE
Non-Null Count   Dtype  
--------------   -----  
307233 non-null  float64
dtypes: float64(1)
memory usage: 2.3 MB
<class 'pandas.Series'>
RangeIndex: 307511 entries, 0 to 307510
Series name: OCCUPATION_TYPE
Non-Null Count   Dtype
--------------   -----
211120 non-null  str  
dtypes: str(1)
memory usage: 2.3 MB


In [23]:
# Ex 2.3
df['NAME_INCOME_TYPE'].value_counts(dropna=False)


df['NAME_INCOME_TYPE'].value_counts(normalize=True, dropna=False)



NAME_INCOME_TYPE
Working                 0.516320
Commercial associate    0.232892
Pensioner               0.180033
State servant           0.070576
Unemployed              0.000072
Student                 0.000059
Businessman             0.000033
Maternity leave         0.000016
Name: proportion, dtype: float64

In [35]:
# Ex 2.4

df[['AGE_YEARS', 'CREDIT_INCOME_RATIO', 'CNT_CHILDREN']].describe()

# 19 Children has to be a record or something

,AGE_YEARS,CREDIT_INCOME_RATIO,CNT_CHILDREN
count,307511.000000,307511.000000,307511.000000
mean,43.906900,3.957570,0.417052
std,11.947950,2.689728,0.722121
min,20.503765,0.004808,0.000000
25%,33.984942,2.018667,0.000000
50%,43.121150,3.265067,0.000000
75%,53.886379,5.159880,1.000000
max,69.073238,84.736842,19.000000


**Your answer (2.1):**

...Yeah could be real - maybe look at that specific borrower details and see if anything jumps out.

## 3 — Selecting columns

Single column → Series. List of columns → DataFrame. This distinction trips up beginners constantly, so drill it.

**Key syntax**

```python
df["AMT_CREDIT"]                       # Series (single bracket, string)
df[["AMT_CREDIT"]]                     # DataFrame (double bracket = list)
df[["AMT_CREDIT", "AMT_ANNUITY"]]      # DataFrame, two columns
df.select_dtypes("number")             # all numeric columns
df.select_dtypes("object")             # all string/categorical columns
```

**Worked example**:

In [ ]:
single = df["AMT_CREDIT"]            # Series
multi  = df[["AMT_CREDIT", "AMT_ANNUITY", "TARGET"]]   # DataFrame
print(type(single), "->", single.shape)
print(type(multi),  "->", multi.shape)
multi.head()


### Exercises

**3.1** — Select just `SK_ID_CURR` and `TARGET` as a DataFrame; show 5 rows.

**3.2** — Select the three external scores `EXT_SOURCE_1/2/3` as a DataFrame and call `.describe()` on the result.

**3.3** — Use `select_dtypes` to grab all `object` (text) columns. How many are there? (`.shape[1]`)

**3.4** — Show, in one expression, the difference between `df["TARGET"]` (Series) and `df[["TARGET"]]` (DataFrame) — print the `type()` of each.

**Interpretation 3.1** — Why does a single-column selection return a Series while a list returns a DataFrame? When does that difference actually bite you (think about what you can `.merge` or concat)?

In [36]:
# Ex 3.1

df[['SK_ID_CURR', 'TARGET']].sample(5)


,SK_ID_CURR,TARGET
99781,215836,0
241227,379317,0
9161,110654,0
167681,294379,0
152794,277089,0


In [38]:
# Ex 3.2

df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].describe()



,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
count,134133.000000,3.068510e+05,246546.000000
mean,0.502130,5.143927e-01,0.510853
std,0.211062,1.910602e-01,0.194844
min,0.014568,8.173617e-08,0.000527
25%,0.334007,3.924574e-01,0.370650
50%,0.505998,5.659614e-01,0.535276
75%,0.675053,6.636171e-01,0.669057
max,0.962693,8.549997e-01,0.896010


In [42]:
# Ex 3.3

df_object = df.select_dtypes(include='object')

df_object.shape

# 16 Columns with type 'obbject'

C:\Users\User\AppData\Local\Temp\ipykernel_15132\3987439738.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_object = df.select_dtypes(include='object')


(307511, 16)

In [46]:
# Ex 3.4

series = df['TARGET']

print(type(series))

df_type = df[['TARGET']]

type(df_type)


<class 'pandas.Series'>


pandas.DataFrame

**Your answer (3.1):**

...Can't merge a saeries with a dataframe, Series has a .name and dataframe has .columns, df['col'] works like dict['key']

## 4 — Row selection: `.loc` and `.iloc`

- `.loc[rows, cols]` — selects by **label** (index value / column name).
- `.iloc[rows, cols]` — selects by **integer position**.

`.loc` is inclusive of both slice ends; `.iloc` is exclusive of the stop (like Python slicing).

**Key syntax**

```python
df.loc[0, "AMT_CREDIT"]                  # label 0, one column
df.loc[0:4, ["AMT_CREDIT", "TARGET"]]    # label slice INCLUSIVE of 4 -> 5 rows
df.iloc[0:5, 0:3]                         # positions, stop EXCLUSIVE -> rows 0..4
df.loc[df["TARGET"] == 1, "AMT_CREDIT"]  # boolean rows + a column (Section 5)
```

**Worked example**:

In [ ]:
print(df.loc[0:2, ["SK_ID_CURR", "AMT_CREDIT", "TARGET"]])   # 3 rows (0,1,2)
print("---")
print(df.iloc[0:2, 0:4])                                     # 2 rows, first 4 cols


### Exercises

**4.1** — With `.loc`, show rows with index 10 through 15 (inclusive) for columns `AGE_YEARS` and `TARGET`. How many rows come back?

**4.2** — With `.iloc`, show the first 3 rows and first 5 columns.

**4.3** — With `.iloc`, grab the *last* 5 rows (hint: negative positions) of `SK_ID_CURR`.

**4.4** — Set `df2 = df.set_index("SK_ID_CURR")`, then use `.loc` to pull the row for client `100002`. (This shows `.loc` selecting by a *meaningful* label, not just 0..N.)

**Interpretation 4.1** — Why is `.loc[0:4]` five rows but `.iloc[0:4]` four? State the rule for each in one line.

In [ ]:
# Ex 4.1

df[['AGE_YEARS', "TARGET"]].loc[10:15]

#  6 rows came back



,AGE_YEARS,TARGET
10,27.917864,0
11,55.898700,0
12,36.793977,0
13,38.565366,0
14,39.926078,0
15,23.895962,0


In [50]:
# Ex 4.2

df.iloc[0:3,0:5]

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR
0,100002,1,Cash loans,M,N
1,100003,0,Cash loans,F,N
2,100004,0,Revolving loans,M,Y


In [56]:
# Ex 4.3


print(df['SK_ID_CURR'].iloc[-5:])




307506    456251
307507    456252
307508    456253
307509    456254
307510    456255
Name: SK_ID_CURR, dtype: int64


In [65]:
# Ex 4.4

df2 = df.set_index('SK_ID_CURR')

df2.loc[100002]


TARGET                           1
NAME_CONTRACT_TYPE      Cash loans
CODE_GENDER                      M
FLAG_OWN_CAR                     N
FLAG_OWN_REALTY                  Y
                           ...    
AMT_MONTHLY_ANNUITY       2058.375
AGE_YEARS                25.902806
YEARS_EMPLOYED            1.744011
CREDIT_INCOME_RATIO       2.007889
ANNUITY_INCOME_RATIO      0.121978
Name: 100002, Length: 126, dtype: object

**Your answer (4.1):**

...

## 5 — Boolean filtering (the analyst's daily bread)

A comparison on a Series returns a **boolean mask**; index a DataFrame with it to keep matching rows. Combine masks with `&` (and), `|` (or), `~` (not) — **each condition in parentheses**. `.query()` is a readable alternative.

**Key syntax**

```python
df[df["AMT_INCOME_TOTAL"] > 200000]
df[(df["CODE_GENDER"] == "F") & (df["CNT_CHILDREN"] >= 2)]     # parentheses required
df[df["NAME_INCOME_TYPE"].isin(["Working", "Pensioner"])]
df[df["AMT_ANNUITY"].between(20000, 40000)]
df.query("AMT_INCOME_TOTAL > 200000 and CODE_GENDER == 'F'")   # same, readable
```

**Worked example** — default rate for a filtered slice (note the chained `.mean()`):

In [ ]:
mask = (df["CODE_GENDER"] == "F") & (df["AMT_INCOME_TOTAL"] > 300000)
sub = df[mask]
print("rows:", len(sub), " default rate:", round(sub["TARGET"].mean(), 4),
      " vs baseline:", round(df["TARGET"].mean(), 4))


### Exercises

**5.1** — Keep rows where `AMT_CREDIT > 1_000_000`. How many? What's their default rate vs baseline?

**5.2** — Clients who own a car **and** realty **and** have >2 children. Count them.

**5.3** — Use `.isin` to keep `NAME_EDUCATION_TYPE` in `["Higher education", "Academic degree"]`. Default rate of that slice?

**5.4** — Rewrite 5.1 using `.query()`.

**5.5** — Rows where `OCCUPATION_TYPE` is missing (`.isna()`). Default rate of the missing-occupation group vs the non-missing group — is missingness itself informative?

**Interpretation 5.1** — Why do the sub-conditions each need parentheses when you use `&`/`|`? (What does Python's operator precedence do without them?)
**Interpretation 5.2** — From 5.5: does whether `OCCUPATION_TYPE` is missing predict default? What does that imply about dropping vs flagging missing values?

In [71]:
# Ex 5.1

df_3 = df[df['AMT_CREDIT'] > 1000000]\

df_3.shape

#  49985 Rows
    

(49985, 127)

In [ ]:
# Ex 5.2

cols = df.columns.tolist()

for col in cols:
    
    if 'car' in col.lower():
        
        print(col)
        
    elif 'realty' in col.lower():
        
        print(col)
    
    elif 'children' in col.lower():
        print(col)
            
    else:
        continue    


df_4 = df[(df['CNT_CHILDREN'] > 2) & (df['FLAG_OWN_CAR'] == 'Y') & (df['FLAG_OWN_REALTY'] == 'Y')]

df_4.shape

# df['FLAG_OWN_CAR'].value_counts()

# 1376 borrowers meet this criteria

FLAG_OWN_CAR
FLAG_OWN_REALTY
CNT_CHILDREN
OWN_CAR_AGE


(1376, 127)

In [ ]:
# Ex 5.3

df['NAME_EDUCATION_TYPE'].value_counts()

df_5 = df[df['NAME_EDUCATION_TYPE'].isin(['Higher education', 'Academic degree'])]

df_5.sample(5)

f'{df_5['TARGET'].mean():.2%}'

# Default Rate of this slice is 5.35%


'5.35%'

In [89]:
# Ex 5.4
df.query('AMT_CREDIT > 1000000')

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,...,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,AMT_MONTHLY_ANNUITY,AGE_YEARS,YEARS_EMPLOYED,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2974.875,45.900068,3.252567,4.790750,0.132217
6,100009,0,Cash loans,F,Y,Y,1,171000.0,1560726.0,41301.0,1395000.0,Unaccompanied,Commercial associate,Higher education,Married,House / apartment,0.035792,-13778,-3130,-1213.0,-619,17.0,1,1,0,1,1,0,Accountants,3.0,...,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0.0,0.0,0.0,1.0,1.0,2.0,3441.750,37.722108,8.569473,9.127053,0.241526
7,100010,0,Cash loans,M,Y,Y,0,360000.0,1530000.0,42075.0,1530000.0,Unaccompanied,State servant,Higher education,Married,House / apartment,0.003122,-18850,-449,-4597.0,-2379,8.0,1,1,1,1,0,0,Managers,2.0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,3506.250,51.608487,1.229295,4.250000,0.116875
8,100011,0,Cash loans,F,N,Y,0,112500.0,1019610.0,33826.5,913500.0,Children,Pensioner,Secondary / secondary special,Married,House / apartment,0.018634,-20099,365243,-7427.0,-3514,NaN,1,0,0,1,0,0,NaN,2.0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,2818.875,55.028063,-999.980835,9.063200,0.300680
21,100025,0,Cash loans,F,Y,Y,1,202500.0,1132573.5,37561.5,927000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Married,House / apartment,0.025164,-14815,-1652,-2299.0,-2299,14.0,1,1,0,1,0,0,Sales staff,3.0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0,3130.125,40.561259,4.522930,5.592956,0.185489
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307487,456231,0,Cash loans,M,N,Y,0,117000.0,1071909.0,31473.0,936000.0,Unaccompanied,Pensioner,Secondary / secondary special,Married,House / apartment,0.010147,-23125,365243,-5485.0,-4115,NaN,1,0,0,1,0,0,NaN,2.0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,8.0,2622.750,63.312799,-999.980835,9.161615,0.269000
307491,456235,0,Cash loans,M,Y,Y,2,90000.0,1078200.0,31522.5,900000.0,Unaccompanied,Commercial associate,Secondary / secondary special,Married,House / apartment,0.019101,-10976,-1953,-5048.0,-3369,15.0,1,1,1,1,0,0,Drivers,4.0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,7.0,2626.875,30.050650,5.347023,11.980000,0.350250
307492,456236,0,Cash loans,M,Y,Y,0,585000.0,1575000.0,43443.0,1575000.0,Unaccompanied,Working,Secondary / secondary special,Married,House / apartment,0.028663,-20965,-1618,-1764.0,-4410,2.0,1,1,0,1,0,0,Sales staff,2.0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,3620.250,57.399042,4.429843,2.692308,0.074262
307498,456242,0,Cash loans,M,Y,Y,0,198000.0,1312110.0,52168.5,1125000.0,Unaccompanied,Commercial associate,Secondary / secondary special,M

In [111]:
# Ex 5.5

missing_occupation = df[df['OCCUPATION_TYPE'].isnull()]

print(f'{missing_occupation['TARGET'].mean():.2%}')

occupation_types = df[df['OCCUPATION_TYPE'].notna()]

print(f'{occupation_types['TARGET'].mean():.2%}')



6.51%
8.79%


In [113]:
print(df.groupby(df['OCCUPATION_TYPE'].isnull())['TARGET'].mean().map('{:.2%}'.format).rename({True: 'Missing', False: 'Not missing'}))

OCCUPATION_TYPE
Not missing    8.79%
Missing        6.51%
Name: TARGET, dtype: str


**Your answer (5.1, 5.2):**

...

## 6 — Creating and transforming columns

Build new columns from vectorised expressions. For conditional logic use `np.where` (binary) or `np.select` (multi-branch) — never a Python loop. `.assign()` adds columns and returns a new frame (chain-friendly).

**Key syntax**

```python
df["ratio"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
df["high"]  = np.where(df["AMT_INCOME_TOTAL"] > 200000, 1, 0)       # binary
df["band"]  = np.select(
    [df["AGE_YEARS"] < 30, df["AGE_YEARS"] < 45, df["AGE_YEARS"] < 60],
    ["<30", "30-44", "45-59"], default="60+")                        # multi-branch
df["clip"]  = df["AMT_INCOME_TOTAL"].clip(upper=1_000_000)           # cap outliers
df2 = df.assign(new_col = lambda d: d["AMT_CREDIT"] * 1.05)          # returns new frame
```

**Worked example**:

In [ ]:
df["CREDIT_PER_PERSON"] = df["AMT_CREDIT"] / df["CNT_FAM_MEMBERS"]
df["HIGH_BURDEN"]       = np.where(df["CREDIT_INCOME_RATIO"] > 4, 1, 0)
df["AGE_BAND"]          = np.select(
    [df["AGE_YEARS"] < 30, df["AGE_YEARS"] < 45, df["AGE_YEARS"] < 60],
    ["<30", "30-44", "45-59"], default="60+")
df[["CREDIT_PER_PERSON", "HIGH_BURDEN", "AGE_BAND"]].head()


### Exercises

**6.1** — Create `GOODS_CREDIT_GAP = AMT_CREDIT - AMT_GOODS_PRICE`. Describe it — are there negative values, and what would negative mean?

**6.2** — Create a binary `IS_PENSIONER = 1` where `NAME_INCOME_TYPE == "Pensioner"`, else 0, using `np.where`. What's the default rate of pensioners vs non?

**6.3** — Build `INCOME_BAND` with `np.select`: `"<100k"`, `"100k-200k"`, `"200k-400k"`, `"400k+"`.

**6.4** — Create `INCOME_CLIPPED = AMT_INCOME_TOTAL` capped at its 99th percentile (`df["AMT_INCOME_TOTAL"].quantile(0.99)`) using `.clip(upper=...)`. Compare `mean` before and after.

**6.5** — Redo 6.2 with `.assign()` so it returns a new frame without mutating `df`.

**Interpretation 6.1** — `np.where` vs `np.select` vs `.apply(lambda ...)`: rank them for speed on 300k rows and say why the loop-like option loses.

In [ ]:
# Ex 6.1



In [ ]:
# Ex 6.2



In [ ]:
# Ex 6.3



In [ ]:
# Ex 6.4



In [ ]:
# Ex 6.5



**Your answer (6.1):**

...

## 7 — Missing data and the sentinel

Detect with `.isna()` / `.notna()`. Handle with `.fillna()`, `.dropna()`, or by turning a **sentinel** into a real `NaN`. Aggregations skip `NaN` by default (`mean` ignores them), which is usually what you want — but not always.

**Key syntax**

```python
df.isna().sum()                          # missing count per column
df.isna().mean().sort_values()           # missing FRACTION per column
df["col"].fillna(0)                      # impute constant
df["col"].fillna(df["col"].median())     # impute median
df["col"].replace(365243, np.nan)        # sentinel -> NaN
df.dropna(subset=["EXT_SOURCE_1"])       # drop rows missing a key col
```

**Worked example** — the `DAYS_EMPLOYED` sentinel, seen and fixed:

In [ ]:
print("sentinel rows:", (df["DAYS_EMPLOYED"] == 365243).sum())
print("dirty mean years:", (-df["DAYS_EMPLOYED"] / 365.25).mean().round(2))

df["YEARS_EMPLOYED_CLEAN"] = (-df["DAYS_EMPLOYED"].replace(365243, np.nan)) / 365.25
df["EMPLOYED_MISSING"]     = df["DAYS_EMPLOYED"].eq(365243).astype(int)  # keep the info!
print("clean mean years: ", df["YEARS_EMPLOYED_CLEAN"].mean().round(2))


### Exercises

**7.1** — Top 10 columns by missing *fraction* (`df.isna().mean().sort_values(ascending=False).head(10)`). Which columns are almost entirely missing?

**7.2** — For `EXT_SOURCE_1`, count missings, then make `EXT_SOURCE_1_FILLED` imputed with the column median. Compare the mean before and after — does median-imputation change it?

**7.3** — Confirm your `YEARS_EMPLOYED_CLEAN` has no value above, say, 60. (`.max()`.)

**7.4** — Compute the default rate for `EMPLOYED_MISSING == 1` vs `== 0`. Is "employment unknown" informative?

**Interpretation 7.1** — Why did we create `EMPLOYED_MISSING` *before* replacing the sentinel? What would median-imputing the sentinel away, with no flag, hide from the model?
**Interpretation 7.2** — When is `fillna(0)` outright wrong? Name a column here where 0 is a real value and imputing 0 would corrupt it.

In [ ]:
# Ex 7.1



In [ ]:
# Ex 7.2



In [ ]:
# Ex 7.3



In [ ]:
# Ex 7.4



**Your answer (7.1, 7.2):**

...

## 8 — Dtypes and memory

Big tables waste memory in the default dtypes. `object` (Python strings) is heavy; **`category`** stores repeated strings as integers. Numeric columns can often be **downcast** (float64→float32, int64→int32). This matters when you're about to load and merge 13M-row installment tables.

**Key syntax**

```python
df.memory_usage(deep=True).sum() / 1e6         # MB, deep counts string bytes
df["NAME_INCOME_TYPE"].astype("category")       # repeated strings -> category
pd.to_numeric(df["CNT_CHILDREN"], downcast="integer")
pd.to_numeric(df["AMT_CREDIT"],   downcast="float")
```

**Worked example**:

In [ ]:
before = df.memory_usage(deep=True).sum() / 1e6
cat_cols = df.select_dtypes("object").columns
df_cat = df.copy()
for c in cat_cols:
    df_cat[c] = df_cat[c].astype("category")
after = df_cat.memory_usage(deep=True).sum() / 1e6
print(f"object->category: {before:.0f} MB -> {after:.0f} MB")


### Exercises

**8.1** — Print the total memory of `df` in MB with `deep=True`.

**8.2** — Convert `NAME_EDUCATION_TYPE` to `category` and compare that single column's `memory_usage(deep=True)` before and after.

**8.3** — Downcast `CNT_CHILDREN` to the smallest integer type with `pd.to_numeric(..., downcast="integer")`. Confirm the new dtype and that values are unchanged.

**8.4** — Load `bureau.csv` with only `usecols=["SK_ID_CURR", "CREDIT_ACTIVE", "AMT_CREDIT_SUM_DEBT"]` and print its memory. (You'll reuse this trick for the big child tables.)

**Interpretation 8.1** — Why does `category` save so much on `NAME_INCOME_TYPE` but little on a near-unique column like `SK_ID_CURR`? What property of the column decides the payoff?

In [ ]:
# Ex 8.1



In [ ]:
# Ex 8.2



In [ ]:
# Ex 8.3



In [ ]:
# Ex 8.4
# bureau_small = pd.read_csv(f"{DATA}/bureau.csv", usecols=[...])



**Your answer (8.1):**

...

## 9 — String methods

The `.str` accessor vectorises Python string ops over a column. Use it to clean, standardise, and extract from text columns.

**Key syntax**

```python
df["ORGANIZATION_TYPE"].str.lower()
df["ORGANIZATION_TYPE"].str.contains("Business")     # boolean mask
df["ORGANIZATION_TYPE"].str.replace(" ", "_")
df["NAME_INCOME_TYPE"].str.startswith("State")
df["ORGANIZATION_TYPE"].str.split(":").str[0]        # take first piece
```

**Worked example**:

In [ ]:
org = df["ORGANIZATION_TYPE"]
print("n containing 'Business':", org.str.contains("Business", na=False).sum())
# strip the trailing "Type N" to get a coarse industry
df["ORG_COARSE"] = org.str.split(":").str[0].str.replace(r" Type \d+", "", regex=True)
df["ORG_COARSE"].value_counts().head(8)


### Exercises

**9.1** — How many rows have `ORGANIZATION_TYPE` containing `"Trade"`? (Use `.str.contains(..., na=False)`.)

**9.2** — Make a lowercase, underscore-joined version of `NAME_FAMILY_STATUS` (e.g. `"Single / not married" -> "single_/_not_married"`).

**9.3** — Boolean mask: `NAME_INCOME_TYPE` starting with `"State"`. Default rate of that group?

**9.4** — From `WEEKDAY_APPR_PROCESS_START`, make a 3-letter uppercase code (`"WEDNESDAY" -> "WED"`) with `.str[:3].str.upper()`.

**Interpretation 9.1** — Why is `.str.contains(..., na=False)` safer than leaving `na` default when the column has missing values?

In [ ]:
# Ex 9.1



In [ ]:
# Ex 9.2



In [ ]:
# Ex 9.3



In [ ]:
# Ex 9.4



**Your answer (9.1):**

...

## 10 — Sorting, ranking, duplicates

**Key syntax**

```python
df.sort_values("AMT_CREDIT", ascending=False).head()
df.sort_values(["NAME_INCOME_TYPE", "AMT_CREDIT"], ascending=[True, False])
df.nlargest(10, "AMT_INCOME_TOTAL")           # faster top-N than full sort
df["AMT_CREDIT"].rank(pct=True)               # percentile rank 0..1
df.drop_duplicates(subset=["SK_ID_CURR"])
df["SK_ID_CURR"].duplicated().any()           # are there dupes?
```

**Worked example**:

In [ ]:
print(df.nlargest(5, "AMT_INCOME_TOTAL")[["SK_ID_CURR", "AMT_INCOME_TOTAL", "TARGET"]])
df["INCOME_PCTL"] = df["AMT_INCOME_TOTAL"].rank(pct=True)
df[["AMT_INCOME_TOTAL", "INCOME_PCTL"]].sort_values("INCOME_PCTL").tail(3)


### Exercises

**10.1** — Top 10 clients by `AMT_CREDIT` with `nlargest`. Show `SK_ID_CURR`, `AMT_CREDIT`, `TARGET`.

**10.2** — Sort by `NAME_INCOME_TYPE` ascending then `AMT_INCOME_TOTAL` descending; show 10 rows.

**10.3** — Add a percentile-rank column for `AMT_ANNUITY` (`rank(pct=True)`). What annuity sits at the 95th percentile? (Filter where the rank ≈ 0.95, or use `.quantile(0.95)` to check.)

**10.4** — Confirm `SK_ID_CURR` is unique in `application_train` (`.duplicated().any()` should be `False`).

**Interpretation 10.1** — Why is `nlargest(10, col)` preferable to `sort_values(col).tail(10)` on a 300k-row frame?

In [ ]:
# Ex 10.1



In [ ]:
# Ex 10.2



In [ ]:
# Ex 10.3



In [ ]:
# Ex 10.4



**Your answer (10.1):**

...

## 11 — Binning: `cut` and `qcut`

- `pd.cut(x, bins)` — **fixed-width / explicit edges** (e.g. age bands 20–30, 30–40...).
- `pd.qcut(x, q)` — **equal-count quantile bins** (e.g. income deciles, credit-score deciles). This is exactly how you bucket a score to validate it.

**Key syntax**

```python
df["AGE_BIN"]   = pd.cut(df["AGE_YEARS"], bins=[20,30,40,50,60,100],
                         labels=["20s","30s","40s","50s","60+"])
df["INC_DECILE"]= pd.qcut(df["AMT_INCOME_TOTAL"], q=10, labels=False, duplicates="drop")
```

**Worked example** — default rate by score decile (the core scorecard check, in pandas):

In [ ]:
tmp = df.dropna(subset=["EXT_SOURCE_2"]).copy()
tmp["SCORE_DECILE"] = pd.qcut(tmp["EXT_SOURCE_2"], q=10, labels=False)
tmp.groupby("SCORE_DECILE")["TARGET"].agg(["size", "mean"])


### Exercises

**11.1** — `pd.cut` `AGE_YEARS` into 5-year-ish bands of your choice; `value_counts` the result.

**11.2** — Default rate by that age band (`groupby(...)["TARGET"].mean()`). Does it fall with age?

**11.3** — `pd.qcut` `AMT_CREDIT` into deciles; default rate per decile.

**11.4** — `pd.qcut` `CREDIT_INCOME_RATIO` into deciles (drop NaN first); default rate per decile. Is the burden→risk relationship monotonic?

**Interpretation 11.1** — `cut` vs `qcut`: when do you want equal-width bins and when equal-count? Why does a scorecard analyst almost always reach for `qcut`?
**Interpretation 11.2** — From 11.3: does default rate change monotonically across credit deciles, or is it flat/non-monotonic? What does a flat relationship tell you about `AMT_CREDIT` as a standalone predictor?

In [ ]:
# Ex 11.1



In [ ]:
# Ex 11.2



In [ ]:
# Ex 11.3



In [ ]:
# Ex 11.4



**Your answer (11.1, 11.2):**

...

## 12 — Frequency & rate tables: `value_counts`, `crosstab`, `pivot_table`

**Key syntax**

```python
df["NAME_INCOME_TYPE"].value_counts(normalize=True)
pd.crosstab(df["NAME_EDUCATION_TYPE"], df["TARGET"])                 # counts
pd.crosstab(df["NAME_EDUCATION_TYPE"], df["TARGET"], normalize="index")  # row %
pd.pivot_table(df, index="NAME_EDUCATION_TYPE", values="TARGET", aggfunc="mean")
pd.pivot_table(df, index="AGE_BAND", columns="CODE_GENDER",
               values="TARGET", aggfunc="mean")                      # two-way rate grid
```

**Worked example** — a two-way default-rate grid:

In [ ]:
grid = pd.pivot_table(df, index="AGE_BAND", columns="CODE_GENDER",
                      values="TARGET", aggfunc="mean")
grid


### Exercises

**12.1** — `crosstab` of `NAME_EDUCATION_TYPE` (rows) vs `TARGET`, normalized by row — this gives the default rate directly in the `1` column.

**12.2** — `pivot_table`: default rate by `NAME_INCOME_TYPE`. Sort descending.

**12.3** — `pivot_table` with `index="NAME_FAMILY_STATUS"`, `columns="CODE_GENDER"`, `values="TARGET"`, `aggfunc="mean"`. Drop the `XNA` gender first.

**12.4** — Add `aggfunc=["mean", "size"]` to a pivot so you see rate **and** group size together. Why do you always want the size alongside the rate?

**Interpretation 12.1** — A `pivot_table` cell can be a tiny, unstable group. What minimum group size would you trust before quoting a cell's default rate, and why does the `size` column from 12.4 guard you?

In [ ]:
# Ex 12.1



In [ ]:
# Ex 12.2



In [ ]:
# Ex 12.3



In [ ]:
# Ex 12.4



**Your answer (12.1):**

...

## 13 — GroupBy I: split-apply-combine

The single most important analyst skill. `groupby` **splits** rows into groups, **applies** an aggregate to each, and **combines** the results. Because `TARGET` is 0/1, `groupby(cat)["TARGET"].mean()` is default-rate-by-category.

**Key syntax**

```python
df.groupby("NAME_EDUCATION_TYPE")["TARGET"].mean()
df.groupby("NAME_EDUCATION_TYPE")["TARGET"].agg(["size", "mean"])
df.groupby(["CODE_GENDER", "NAME_EDUCATION_TYPE"])["TARGET"].mean()   # multi-key
df.groupby("NAME_INCOME_TYPE")["AMT_CREDIT"].mean().sort_values()
```

**Worked example**:

In [ ]:
(df.groupby("NAME_EDUCATION_TYPE")["TARGET"]
   .agg(n="size", default_rate="mean")
   .sort_values("default_rate", ascending=False))


### Exercises

**13.1** — Default rate and count by `NAME_INCOME_TYPE`, sorted by rate. (`.agg(n="size", rate="mean")`.)

**13.2** — Mean `AMT_CREDIT` by `NAME_FAMILY_STATUS`.

**13.3** — Multi-key: default rate by `(CODE_GENDER, NAME_EDUCATION_TYPE)`. Drop `XNA` gender.

**13.4** — Default rate by `OCCUPATION_TYPE`, but keep only groups with ≥5000 rows (compute `.agg(["size","mean"])` then filter the result with a boolean mask on `size`). Sort descending.

**13.5** — Mean of `EXT_SOURCE_2` by `TARGET` (0 vs 1). Is the score meaningfully lower for defaulters?

**Interpretation 13.1** — From 13.4: two highest- and two lowest-risk occupations. What links the risky ones?
**Interpretation 13.2** — Why filter to ≥5000 rows before ranking occupations by default rate? What artefact appears if you don't?

In [ ]:
# Ex 13.1



In [ ]:
# Ex 13.2



In [ ]:
# Ex 13.3



In [ ]:
# Ex 13.4



In [ ]:
# Ex 13.5



**Your answer (13.1, 13.2):**

...

## 14 — GroupBy II: named aggregation & many columns at once

Named aggregation (`func=("col", "aggfunc")`) gives clean output column names and lets you aggregate several columns with different functions in one call — exactly what you do when summarising a child table into features.

**Key syntax**

```python
df.groupby("NAME_INCOME_TYPE").agg(
    n            = ("TARGET", "size"),
    default_rate = ("TARGET", "mean"),
    mean_credit  = ("AMT_CREDIT", "mean"),
    max_income   = ("AMT_INCOME_TOTAL", "max"),
)

# same aggfunc across many columns:
df.groupby("CODE_GENDER")[["AMT_CREDIT", "AMT_ANNUITY"]].agg(["mean", "median"])
```

**Worked example**:

In [ ]:
df.groupby("NAME_EDUCATION_TYPE").agg(
    n            = ("TARGET", "size"),
    default_rate = ("TARGET", "mean"),
    mean_credit  = ("AMT_CREDIT", "mean"),
    med_income   = ("AMT_INCOME_TOTAL", "median"),
).sort_values("default_rate", ascending=False)


### Exercises

**14.1** — By `NAME_FAMILY_STATUS`, produce a table with: count, default rate, mean credit, mean age.

**14.2** — By `NAME_CONTRACT_TYPE`, aggregate `AMT_CREDIT` and `AMT_ANNUITY` with both `mean` and `median` in one call.

**14.3** — By `AGE_BAND`, produce default rate and mean `EXT_SOURCE_2`. Do older bands have higher scores *and* lower default?

**14.4** — Build a custom aggregate: by `NAME_INCOME_TYPE`, the fraction of clients with `CREDIT_INCOME_RATIO > 4` (hint: aggregate a boolean with `"mean"`).

**Interpretation 14.1** — Named aggregation produces one row per group with many feature columns. Connect this to the SQL worksheet: what SQL clause is `.agg(name=(col, func))` the direct equivalent of?

In [ ]:
# Ex 14.1



In [ ]:
# Ex 14.2



In [ ]:
# Ex 14.3



In [ ]:
# Ex 14.4



**Your answer (14.1):**

...

## 15 — GroupBy III: `transform`, `filter`, `apply`

- `.transform()` returns a result **aligned to the original rows** (same length) — perfect for group-level features and group-based imputation.
- `.filter()` keeps/drops whole groups by a group-level condition.
- `.apply()` is the flexible, slower escape hatch for arbitrary per-group logic.

**Key syntax**

```python
# group mean attached to every row:
df["ed_mean_credit"] = df.groupby("NAME_EDUCATION_TYPE")["AMT_CREDIT"].transform("mean")
# how far each client is from their education group's mean:
df["credit_vs_ed"]   = df["AMT_CREDIT"] - df["ed_mean_credit"]
# group-median imputation (a real technique):
df["ext2_imp"] = df["EXT_SOURCE_2"].fillna(
    df.groupby("NAME_EDUCATION_TYPE")["EXT_SOURCE_2"].transform("median"))
# keep only large groups:
df.groupby("OCCUPATION_TYPE").filter(lambda g: len(g) >= 5000)
```

**Worked example**:

In [ ]:
df["ED_MEAN_CREDIT"] = df.groupby("NAME_EDUCATION_TYPE")["AMT_CREDIT"].transform("mean")
df["CREDIT_VS_ED"]   = df["AMT_CREDIT"] - df["ED_MEAN_CREDIT"]
df[["NAME_EDUCATION_TYPE", "AMT_CREDIT", "ED_MEAN_CREDIT", "CREDIT_VS_ED"]].head()


### Exercises

**15.1** — Attach each client's income-type mean income as `INCOME_TYPE_MEAN` with `transform("mean")`, then `INCOME_VS_TYPE = AMT_INCOME_TOTAL - INCOME_TYPE_MEAN`.

**15.2** — Group-median-impute `EXT_SOURCE_3` by `NAME_EDUCATION_TYPE`. Confirm the number of missing drops to 0 (or near it).

**15.3** — Use `transform` to compute each client's income as a **fraction of their income-type max** (`x / group_max`).

**15.4** — With `.filter`, keep only `ORGANIZATION_TYPE` groups having ≥2000 rows; how many rows remain?

**Interpretation 15.1** — `agg` vs `transform`: both take a group aggregate, but one returns one-row-per-group and the other one-row-per-original-row. Why is `transform` the right tool for *making a feature*, and `agg` the right tool for a *summary table*?
**Interpretation 15.2** — Group-median imputation (15.2) can subtly leak information if done wrong in a train/test split. In one sentence, what's the danger and when should the medians be computed?

In [ ]:
# Ex 15.1



In [ ]:
# Ex 15.2



In [ ]:
# Ex 15.3



In [ ]:
# Ex 15.4



**Your answer (15.1, 15.2):**

...

## 16 — Reshaping: melt, pivot, stack/unstack, MultiIndex

- **wide → long**: `df.melt(...)`.
- **long → wide**: `df.pivot(...)` / `pivot_table`.
- **MultiIndex**: a groupby on 2+ keys yields a hierarchical index; `.unstack()` turns an index level into columns, `.stack()` does the reverse.

**Key syntax**

```python
g = df.groupby(["AGE_BAND", "CODE_GENDER"])["TARGET"].mean()   # MultiIndex Series
g.unstack()                                # CODE_GENDER becomes columns -> a grid
g.unstack().stack()                        # back to long
long = df.melt(id_vars="SK_ID_CURR",
               value_vars=["EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3"])
```

**Worked example**:

In [ ]:
g = df[df.CODE_GENDER != "XNA"].groupby(["AGE_BAND", "CODE_GENDER"])["TARGET"].mean()
print(type(g.index))            # MultiIndex
g.unstack()                     # -> M / F columns


### Exercises

**16.1** — Group default rate by `(NAME_EDUCATION_TYPE, CODE_GENDER)` (drop XNA), then `.unstack()` the gender level into columns. Which education row shows the widest M–F gap?

**16.2** — From that MultiIndex Series, use `.loc[("Higher education",)]` to pull just that education level's rates.

**16.3** — `melt` the three `EXT_SOURCE_*` columns into long form (`id_vars="SK_ID_CURR"`), then `groupby("variable")["value"].mean()` to get each source's overall mean in one shot.

**16.4** — Take the unstacked grid from 16.1 and `.stack()` it back; confirm you recover the original long Series.

**Interpretation 16.1** — Why is "long" (melted) form often better for plotting and grouping, while "wide" form is better for reading and for a model matrix? Give the one-line rule of thumb.

In [ ]:
# Ex 16.1



In [ ]:
# Ex 16.2



In [ ]:
# Ex 16.3



In [ ]:
# Ex 16.4



**Your answer (16.1):**

...

## 17 — Merging I: joining tables

`pd.merge` is pandas' JOIN. `how=` controls which keys survive: `"inner"` (matches only), `"left"` (keep all left rows — the analyst default), `"right"`, `"outer"`. Use `validate=` to assert the relationship and `indicator=True` to see where each row matched.

**Key syntax**

```python
pd.merge(left, right, on="SK_ID_CURR", how="left")
pd.merge(app, agg, on="SK_ID_CURR", how="left",
         validate="one_to_one", indicator=True)
left.merge(right, on="SK_ID_CURR", how="left")     # method form, chain-friendly
```

**Worked example** — merge a per-client bureau summary onto applications (aggregate first — Section 18 covers the aggregation; here just see the merge mechanics):

In [ ]:
bureau = pd.read_csv(f"{DATA}/bureau.csv",
                     usecols=["SK_ID_CURR", "AMT_CREDIT_SUM_DEBT"])
bureau_agg = (bureau.groupby("SK_ID_CURR")["AMT_CREDIT_SUM_DEBT"]
              .sum().rename("TOTAL_BUREAU_DEBT").reset_index())

merged = df[["SK_ID_CURR", "TARGET"]].merge(bureau_agg, on="SK_ID_CURR",
                                            how="left", indicator=True)
print(merged["_merge"].value_counts())     # left_only = clients with no bureau history
merged.head()


### Exercises

**17.1** — Redo the merge above and count how many clients are `left_only` (no bureau record). What fraction of the portfolio?

**17.2** — After the left merge, `TOTAL_BUREAU_DEBT` is `NaN` for no-history clients. Fill with 0 **and** add a `HAS_BUREAU` flag (1/0). Default rate for `HAS_BUREAU==1` vs `==0`?

**17.3** — Merge with `validate="one_to_one"` — confirm it passes (your aggregate is one row per client). Then deliberately try `validate="one_to_many"` and read the error to understand what it checks.

**17.4** — Band `TOTAL_BUREAU_DEBT` (after 0-fill) into deciles with `qcut` (drop zeros or use `duplicates="drop"`) and show default rate per decile.

**Interpretation 17.1** — Why is `how="left"` the right default here rather than `"inner"`? What population would an inner join silently drop, and how would that bias your default-rate estimates?
**Interpretation 17.2** — What does `validate="one_to_one"` protect you from? Describe the silent bug that appears if your "aggregate" accidentally has duplicate `SK_ID_CURR` rows.

In [ ]:
# Ex 17.1



In [ ]:
# Ex 17.2



In [ ]:
# Ex 17.3



In [ ]:
# Ex 17.4



**Your answer (17.1, 17.2):**

...

## 18 — Merging II: aggregate-then-merge (the feature pattern)

The pandas twin of the SQL worksheet's key pattern. A child table is many-rows-per-client; you **must** collapse it to one-row-per-client *before* merging, or you fan out the applications and corrupt every downstream stat.

**The recipe**

```python
child_agg = (child.groupby("SK_ID_CURR")
                  .agg(feat_a=("col1","mean"), feat_b=("col2","sum"))
                  .reset_index())
app = app.merge(child_agg, on="SK_ID_CURR", how="left")
```

**Worked example** — build several bureau features at once and merge them on:

In [ ]:
bureau = pd.read_csv(f"{DATA}/bureau.csv",
    usecols=["SK_ID_CURR","CREDIT_ACTIVE","AMT_CREDIT_SUM_DEBT","AMT_CREDIT_MAX_OVERDUE"])
bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    N_BUREAU        = ("CREDIT_ACTIVE", "size"),
    N_ACTIVE_BUREAU = ("CREDIT_ACTIVE", lambda s: (s == "Active").sum()),
    TOTAL_DEBT      = ("AMT_CREDIT_SUM_DEBT", "sum"),
    MAX_OVERDUE     = ("AMT_CREDIT_MAX_OVERDUE", "max"),
).reset_index()

app = df.merge(bureau_agg, on="SK_ID_CURR", how="left")
print("rows unchanged:", len(app) == len(df))     # must be True (no fan-out)
app[["SK_ID_CURR","TARGET","N_BUREAU","N_ACTIVE_BUREAU","TOTAL_DEBT"]].head()


### Exercises

**18.1** — From `previous_application.csv` (use `usecols=["SK_ID_CURR","NAME_CONTRACT_STATUS","AMT_APPLICATION"]`), build per-client `N_PREV` and `N_REFUSED` (count where status == "Refused"). Merge onto `app`. Confirm row count is unchanged.

**18.2** — Add `REFUSAL_RATIO = N_REFUSED / N_PREV` (guard divide-by-zero). Band it and show default rate per band.

**18.3** — From `installments_payments.csv` (`usecols=["SK_ID_CURR","DAYS_INSTALMENT","DAYS_ENTRY_PAYMENT","AMT_INSTALMENT","AMT_PAYMENT"]`), build per-client `N_LATE = sum(DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT)` and `AVG_PAY_GAP = mean(AMT_INSTALMENT - AMT_PAYMENT)`. Merge on. (This table is ~13M rows — the read takes a moment.)

**18.4** — Default rate for clients with `N_LATE > 0` vs `N_LATE == 0` (0-fill the NaNs first).

**Interpretation 18.1** — Explain, in grain terms, what goes wrong if you `merge` raw `installments_payments` onto `app` *without* aggregating first. What happens to `len(app)` and to `app["TARGET"].mean()`?
**Interpretation 18.2** — After left-merging, missing child data becomes `NaN`. Argue when 0-fill is right (e.g. `N_LATE`) and when it's wrong (e.g. a *ratio* or a *mean*), using one example of each from this section.

In [ ]:
# Ex 18.1



In [ ]:
# Ex 18.2



In [ ]:
# Ex 18.3



In [ ]:
# Ex 18.4



**Your answer (18.1, 18.2):**

...

## 19 — Concatenation

`pd.concat` stacks frames: `axis=0` (rows, e.g. train + test) or `axis=1` (columns, aligning on index). Different from merge — concat aligns on the index/position, it doesn't match on a key.

**Key syntax**

```python
pd.concat([df_a, df_b], axis=0, ignore_index=True)      # stack rows
pd.concat([frame1, frame2], axis=1)                     # side by side on index
```

**Worked example** — stack train and test into one frame with a source flag:

In [ ]:
test = pd.read_csv(f"{DATA}/application_test.csv", usecols=["SK_ID_CURR","AMT_CREDIT"])
train_small = df[["SK_ID_CURR","AMT_CREDIT"]].copy()
train_small["SOURCE"] = "train"
test["SOURCE"] = "test"
both = pd.concat([train_small, test], axis=0, ignore_index=True)
both["SOURCE"].value_counts()


### Exercises

**19.1** — Concatenate `application_train` and `application_test` on the shared columns only (test has no `TARGET`). Add a `SOURCE` column. How many total rows?

**19.2** — Build two small summary frames (e.g. default rate by gender, and count by gender) and `pd.concat(..., axis=1)` them side by side.

**Interpretation 19.1** — When would you `concat(axis=0)` train+test before feature engineering, and what's the one thing you must be careful *not* to do while they're combined (think leakage)?

In [ ]:
# Ex 19.1



In [ ]:
# Ex 19.2



**Your answer (19.1):**

...

## 20 — Window & rolling operations on time series

The child tables are time series (installments, monthly balances). Sort within each loan/client, then use `groupby(...).cumsum()`, `.rolling()`, `.expanding()`, or `.shift()` (lag) to build trend features. The pandas twin of SQL window functions.

**Key syntax**

```python
g = inst.sort_values(["SK_ID_PREV", "NUM_INSTALMENT_NUMBER"])
g["running_paid"] = g.groupby("SK_ID_PREV")["AMT_PAYMENT"].cumsum()
g["prev_payment"] = g.groupby("SK_ID_PREV")["AMT_PAYMENT"].shift(1)       # lag
g["mov_avg3"]     = (g.groupby("SK_ID_PREV")["AMT_PAYMENT"]
                       .transform(lambda s: s.rolling(3, min_periods=1).mean()))
```

**Worked example** — running total and lag for one prior loan:

In [ ]:
inst = pd.read_csv(f"{DATA}/installments_payments.csv",
    usecols=["SK_ID_PREV","NUM_INSTALMENT_NUMBER","DAYS_INSTALMENT",
             "DAYS_ENTRY_PAYMENT","AMT_PAYMENT"])
one = inst[inst["SK_ID_PREV"] == inst["SK_ID_PREV"].iloc[0]].sort_values("NUM_INSTALMENT_NUMBER").copy()
one["RUNNING_PAID"] = one["AMT_PAYMENT"].cumsum()
one["PREV_PAYMENT"] = one["AMT_PAYMENT"].shift(1)
one["LATE_DAYS"]    = one["DAYS_ENTRY_PAYMENT"] - one["DAYS_INSTALMENT"]
one.head(10)


### Exercises

**20.1** — For that one loan, add a 3-installment rolling mean of `AMT_PAYMENT` (`.rolling(3, min_periods=1).mean()`).

**20.2** — Across the whole `inst` table, build a per-client feature: number of late installments (`DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT`) via `groupby("SK_ID_CURR")`. (Load `SK_ID_CURR` too.) Top 10 clients by late count.

**20.3** — Per client, the *maximum* `LATE_DAYS` ever recorded. Merge onto `app` and show default rate for `MAX_LATE_DAYS > 30` vs not.

**20.4** — Using `pos_cash_balance.csv` (`usecols=["SK_ID_CURR","SK_DPD"]`), build per-client `MAX_DPD = max(SK_DPD)`. Merge and relate to default.

**Interpretation 20.1** — Rolling and lag features can leak the future. In these `DAYS_*` tables, how do you know a record happened *before* the application, and why does that sign convention protect you from leakage?
**Interpretation 20.2** — Which single time-series feature from this section do you expect to be the strongest default predictor, and what's the credit intuition?

In [ ]:
# Ex 20.1



In [ ]:
# Ex 20.2



In [ ]:
# Ex 20.3



In [ ]:
# Ex 20.4



**Your answer (20.1, 20.2):**

...

## 21 — Method chaining & readable pipelines

Analyst code lives or dies on readability. Chaining with `.assign`, `.query`, `.pipe` lets a transformation read top-to-bottom with no throwaway intermediate variables and no `SettingWithCopyWarning`.

**Key syntax**

```python
result = (df
    .query("AMT_INCOME_TOTAL > 0")
    .assign(cir = lambda d: d["AMT_CREDIT"] / d["AMT_INCOME_TOTAL"],
            age = lambda d: -d["DAYS_BIRTH"] / 365.25)
    .groupby(pd.cut(lambda_placeholder))   # (usually cut a column defined above)
)

def add_ratios(d):                 # reusable step for .pipe
    return d.assign(cir = d["AMT_CREDIT"] / d["AMT_INCOME_TOTAL"])
df.pipe(add_ratios)
```

**Worked example** — a full mini-pipeline, no intermediate variables:

In [ ]:
summary = (df
    .query("AMT_INCOME_TOTAL > 0 and CODE_GENDER != 'XNA'")
    .assign(cir      = lambda d: d["AMT_CREDIT"] / d["AMT_INCOME_TOTAL"],
            age_band = lambda d: pd.cut(-d["DAYS_BIRTH"]/365.25,
                                        bins=[20,30,40,50,60,100]))
    .groupby("age_band", observed=True)
    .agg(n=("TARGET","size"), default_rate=("TARGET","mean"),
         mean_cir=("cir","mean"))
)
summary


### Exercises

**21.1** — Write a single chained pipeline that: filters to `NAME_CONTRACT_TYPE == "Cash loans"`, adds `annuity_income = AMT_ANNUITY / AMT_INCOME_TOTAL`, and returns default rate by `NAME_EDUCATION_TYPE` sorted descending — no intermediate variables.

**21.2** — Write a reusable function `add_core_features(d)` that adds `AGE_YEARS`, `CREDIT_INCOME_RATIO`, and `ANNUITY_INCOME_RATIO`, and apply it with `.pipe`. Confirm the columns appear.

**21.3** — Chain `.query` + `.assign` + `.groupby` to get, for clients under 30, the mean `EXT_SOURCE_2` by `CODE_GENDER`.

**Interpretation 21.1** — Chaining avoids the `SettingWithCopyWarning`. In one or two sentences, what causes that warning, and why does building a new frame via `.assign` sidestep it?

In [ ]:
# Ex 21.1



In [ ]:
# Ex 21.2



In [ ]:
# Ex 21.3



**Your answer (21.1):**

...

## 22 — Performance & correctness

Three habits that separate shaky notebooks from reliable ones on big data.

1. **Vectorise, don't `.apply` row-wise.** `.apply(axis=1)` is a Python loop in disguise — orders of magnitude slower than column arithmetic or `np.select`.
2. **Use `category` dtype** for repeated strings before groupby/merge — faster and lighter.
3. **Avoid chained assignment.** `df[df.x>0]["y"] = 1` may silently fail (`SettingWithCopyWarning`). Use `.loc[mask, "y"] = 1` or build with `.assign`.

**Worked example** — same computation, vectorised vs row-apply (time them):

In [ ]:
import time
# slow: row-wise apply
t0 = time.time()
_ = df.apply(lambda r: r["AMT_CREDIT"] / r["AMT_INCOME_TOTAL"], axis=1)
t_apply = time.time() - t0
# fast: vectorised
t0 = time.time()
_ = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
t_vec = time.time() - t0
print(f"row-apply: {t_apply:.3f}s   vectorised: {t_vec:.4f}s   speedup ~{t_apply/max(t_vec,1e-6):.0f}x")


### Exercises

**22.1** — Reproduce the timing above for `AMT_ANNUITY * 12`: `.apply(axis=1)` vs vectorised. Report the speedup.

**22.2** — Correct way to set values: using `.loc`, set a new column `RISK_FLAG = 1` only for rows where `CREDIT_INCOME_RATIO > 4` (else 0). Do it without triggering a warning.

**22.3** — Convert all `object` columns to `category` and time a `groupby("ORGANIZATION_TYPE")["TARGET"].mean()` before vs after. Any speed difference?

**22.4** — Show the wrong pattern (`df[df.x>0]["y"] = ...`) in a comment and explain in words why it can silently no-op. (Don't actually rely on it.)

**Interpretation 22.1** — You have a 13M-row installments table to feature-engineer. List the three habits from this section in the order they'll save you the most time/pain, and why.

In [ ]:
# Ex 22.1



In [ ]:
# Ex 22.2



In [ ]:
# Ex 22.3



In [ ]:
# Ex 22.4



**Your answer (22.1):**

...

## 23 — Capstone: build a model-ready feature matrix

Everything above, assembled into what a credit-risk analyst actually hands off: **one row per `SK_ID_CURR`**, `TARGET` attached, and a wide set of engineered features from every table. This is the exact twin of the SQL worksheet capstone — build both and diff the results; they should agree.

**The brief** — produce a DataFrame `features` (index or column `SK_ID_CURR`, plus `TARGET`) containing at least:

From **`application_train`**: `AGE_YEARS`, `YEARS_EMPLOYED_CLEAN` (sentinel→NaN), `EMPLOYED_MISSING` flag, `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `EXT_SOURCE_1/2/3`.

From **`bureau`** (aggregate-then-merge): `N_BUREAU`, `N_ACTIVE_BUREAU`, `TOTAL_BUREAU_DEBT`, `MAX_BUREAU_OVERDUE`.

From **`previous_application`**: `N_PREV`, `N_REFUSED`, `REFUSAL_RATIO`.

From **`installments_payments`**: `N_INSTALLMENTS`, `N_LATE_PAYMENTS`, `AVG_PAYMENT_GAP`.

From **`pos_cash_balance`**: `MAX_DPD`.

Requirements: aggregate each child to one-row-per-client *before* merging; `how="left"` so no client is dropped; decide and document your NaN policy per feature (0-fill counts, leave ratios/means NaN or median-impute with a flag); guard divides. Then **validate**: (a) `len(features) == len(df)`; (b) `features["SK_ID_CURR"].is_unique`; (c) `qcut` the table by `EXT_SOURCE_2` and confirm the default-rate gradient survives; (d) `features.isna().mean().sort_values(ascending=False).head()` to eyeball residual missingness.

**Then write the analysis** (markdown below): pick your 5 strongest features, each as *observation → decision*. Example: "`N_LATE_PAYMENTS` default rate climbs from ~7% (0 late) to ~15% (many) → keep; 0-fill no-history clients but add a `HAS_INSTALLMENTS` flag so the model can tell 'never borrowed' from 'borrowed, always on time'."

**Leakage caution:** every feature must be knowable at application time. Note which columns you'd audit (anything dated after `DAYS_DECISION` / the application).

In [ ]:
# Capstone — build `features`.
# Develop each child aggregate in its own scratch cell first, then assemble.
# Skeleton:

# features = df[[
#     "SK_ID_CURR","TARGET","AGE_YEARS","YEARS_EMPLOYED_CLEAN","EMPLOYED_MISSING",
#     "CREDIT_INCOME_RATIO","ANNUITY_INCOME_RATIO",
#     "EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3",
# ]].copy()

# bureau_agg = (...).reset_index()
# prev_agg   = (...).reset_index()
# inst_agg   = (...).reset_index()
# pos_agg    = (...).reset_index()

# for agg in [bureau_agg, prev_agg, inst_agg, pos_agg]:
#     features = features.merge(agg, on="SK_ID_CURR", how="left")

# ... NaN policy, derived ratios/flags ...


In [ ]:
# Validation (a) row count, (b) uniqueness



In [ ]:
# Validation (c) default-rate gradient by EXT_SOURCE_2 decile on `features`



In [ ]:
# Validation (d) residual missingness



**Capstone decision statements (pick 5 features):**

Feature 1:

Feature 2:

Feature 3:

Feature 4:

Feature 5:

NaN policy summary (per feature: 0-fill / median+flag / leave NaN):

Leakage check — columns I'd audit:

## 24 — Self-test: can you answer these cold?

Close the notes. If any are shaky, redo that section.

1. Series vs DataFrame; and why `df["c"]` differs from `df[["c"]]`.
2. `.loc` vs `.iloc` — the label/position rule and the slice-inclusivity difference.
3. Why does the mean of `TARGET` equal the default rate, and where does that shortcut show up in every groupby you wrote?
4. `np.where` vs `np.select` vs row-wise `.apply` — pick one for a 3-branch rule on 300k rows and justify.
5. The `DAYS_EMPLOYED` sentinel: value, detection, and why you add a missing-flag before replacing it.
6. `cut` vs `qcut` — which for a scorecard decile check, and why.
7. `groupby.agg` vs `groupby.transform` — output shape of each and when you use which.
8. The aggregate-then-merge pattern: why aggregating the child table first is non-negotiable (say it in grain terms).
9. `how="left"` vs `"inner"` when attaching bureau features — which population does inner drop, and how does that bias a default rate?
10. One feature you'd audit for leakage in the capstone, and how the `DAYS_*` sign convention helps.

**When you're done:** bring the completed workbook — code and written answers — back for marking. Your capstone `features` frame is the same object your SQL `client_features` view produces; comparing them is the best possible check that both skills are solid.